# Week 1 — Experiment Tracking & Model Registry

### Build your own mini-MLflow before you ever pip-install the real one

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why ad-hoc experiment logging fails at scale.
2. Implement a file-backed tracking store that logs params, metrics, and artifacts.
3. Build a model registry with versioning and stage transitions (Staging/Production).
4. Query and compare runs to pick a 'best' model programmatically.
5. Map every component you built onto MLflow's architecture.

## Prerequisites

- Week 0 (content hashing, run manifests).
- Basic file I/O and JSON.

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_fsrlwtb7


## 1. 🧠 The problem: experiments multiply, memory doesn't

You run 5 experiments before lunch. Each has different hyperparameters, a
different metric, maybe a different dataset slice. By Friday you have 80 runs and
a `results_final_v3_REAL_actually.csv`. Three weeks later someone asks *"which
config gave the 0.91 F1?"* and nobody knows.

Experiment tracking solves a deceptively simple data-management problem:

> For every training run, durably record **what went in** (params, data, code),
> **what came out** (metrics, artifacts), and **enough metadata to find it again**.

A model registry then sits on top to answer the *operational* question:

> Of all these runs, **which model is allowed to serve production traffic right
> now**, and how do I promote/demote/roll back?

We'll build both, file-backed, in this notebook.

## 2. 🔧 From scratch: the tracking store

The data model is small. A **run** belongs to an **experiment** and holds params,
metrics (which can be logged multiple times — think loss per epoch), tags, and
artifacts (files). We persist everything as JSON + files on disk, exactly as
MLflow's local backend does.

In [2]:
@dataclass
class Run:
    run_id: str
    experiment: str
    status: str = "RUNNING"
    params: dict = field(default_factory=dict)
    # metrics map name -> list of (step, value, timestamp) so we keep history
    metrics: dict = field(default_factory=dict)
    tags: dict = field(default_factory=dict)
    artifacts: list = field(default_factory=list)
    start_time: float = field(default_factory=time.time)
    end_time: float | None = None


class TrackingStore:
    """A minimal, file-backed experiment tracker (your mini-MLflow backend)."""

    def __init__(self, root: pathlib.Path):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    # --- internal helpers ---
    def _run_dir(self, run_id): return self.root / "runs" / run_id
    def _run_file(self, run_id): return self._run_dir(run_id) / "run.json"

    def _save(self, run: Run):
        d = self._run_dir(run.run_id)
        (d / "artifacts").mkdir(parents=True, exist_ok=True)
        with open(self._run_file(run.run_id), "w") as f:
            json.dump(asdict(run), f, indent=2)

    def _load(self, run_id) -> Run:
        with open(self._run_file(run_id)) as f:
            return Run(**json.load(f))

    # --- public API (mirrors mlflow.* roughly) ---
    def start_run(self, experiment: str, params: dict | None = None) -> str:
        run_id = hashlib.sha256(f"{time.time()}{random.random()}".encode()).hexdigest()[:12]
        run = Run(run_id=run_id, experiment=experiment, params=params or {})
        self._save(run)
        return run_id

    def log_param(self, run_id, key, value):
        run = self._load(run_id); run.params[key] = value; self._save(run)

    def log_metric(self, run_id, key, value, step=0):
        run = self._load(run_id)
        run.metrics.setdefault(key, []).append([step, float(value), time.time()])
        self._save(run)

    def log_artifact(self, run_id, name: str, content: bytes):
        run = self._load(run_id)
        path = self._run_dir(run_id) / "artifacts" / name
        path.write_bytes(content)
        run.artifacts.append(name); self._save(run)

    def set_tag(self, run_id, key, value):
        run = self._load(run_id); run.tags[key] = value; self._save(run)

    def end_run(self, run_id, status="FINISHED"):
        run = self._load(run_id); run.status = status; run.end_time = time.time(); self._save(run)

    def get_run(self, run_id) -> Run:
        return self._load(run_id)

    def search_runs(self, experiment: str | None = None) -> list[Run]:
        runs = []
        rdir = self.root / "runs"
        if rdir.exists():
            for d in rdir.iterdir():
                r = self._load(d.name)
                if experiment is None or r.experiment == experiment:
                    runs.append(r)
        return runs


store = TrackingStore(WORK / "mlruns")
print("Tracking store at:", store.root)

Tracking store at: /tmp/mlops_week_fsrlwtb7/mlruns


## 3. 🔧 Using the tracker: a real (tiny) training loop

Let's train something with actual learning dynamics so the metric history is
meaningful. We'll fit a linear regression with gradient descent — and log loss at
every step, plus the final params and the learned weights as an artifact.

In [3]:
# Synthetic regression problem: y = 3x + 2 + noise
rng = np.random.RandomState(0)
X = rng.rand(200, 1)
y = 3 * X[:, 0] + 2 + 0.1 * rng.randn(200)

def train_linear(lr: float, epochs: int, run_id: str, store: TrackingStore):
    w, b = 0.0, 0.0
    n = len(X)
    for epoch in range(epochs):
        pred = w * X[:, 0] + b
        err = pred - y
        loss = float(np.mean(err ** 2))
        # gradients
        gw = float(2 * np.mean(err * X[:, 0]))
        gb = float(2 * np.mean(err))
        w -= lr * gw
        b -= lr * gb
        store.log_metric(run_id, "mse", loss, step=epoch)
    # log final outcome
    store.log_metric(run_id, "final_mse", loss, step=epochs)
    store.log_artifact(run_id, "weights.json",
                       json.dumps({"w": w, "b": b}).encode())
    return w, b, loss

# Run a small hyperparameter sweep over learning rate
for lr in [0.05, 0.2, 0.5]:
    rid = store.start_run("linreg_sweep", params={"lr": lr, "epochs": 100})
    store.set_tag(rid, "owner", "course")
    w, b, final = train_linear(lr, 100, rid, store)
    store.end_run(rid)
    print(f"lr={lr:<4}  ->  w={w:.3f} b={b:.3f}  final_mse={final:.5f}  run={rid}")

lr=0.05  ->  w=2.183 b=2.424  final_mse=0.05865  run=e2890897e7f3
lr=0.2   ->  w=2.845 b=2.070  final_mse=0.01020  run=be79a4ebea9c
lr=0.5   ->  w=2.954 b=2.012  final_mse=0.00913  run=20b1a6839cbd


## 4. 🔧 Querying runs: picking a winner programmatically

The whole point of logging is to *query* later. Let's find the best run by final
MSE — the same operation a CI pipeline performs to decide what to promote.

In [4]:
def best_run(store, experiment, metric, mode="min"):
    runs = store.search_runs(experiment)
    def score(r):
        hist = r.metrics.get(metric, [])
        return hist[-1][1] if hist else (float("inf") if mode == "min" else float("-inf"))
    return min(runs, key=score) if mode == "min" else max(runs, key=score)

champion = best_run(store, "linreg_sweep", "final_mse", mode="min")
print("Best run:", champion.run_id)
print("  params :", champion.params)
print("  final_mse:", champion.metrics["final_mse"][-1][1])

# Build a comparison table by hand (this is what the MLflow UI renders)
print("\nComparison table")
print(f"{'run_id':<14}{'lr':<8}{'final_mse':<12}")
for r in store.search_runs("linreg_sweep"):
    lr = r.params.get("lr")
    fm = r.metrics.get("final_mse", [[None, None]])[-1][1]
    print(f"{r.run_id:<14}{str(lr):<8}{fm:<12.6f}")

Best run: 20b1a6839cbd
  params : {'lr': 0.5, 'epochs': 100}
  final_mse: 0.009131032221831968

Comparison table
run_id        lr      final_mse   
20b1a6839cbd  0.5     0.009131    
e2890897e7f3  0.05    0.058650    
be79a4ebea9c  0.2     0.010196    


## 5. 🧠 Tracking ≠ registry

The tracking store records *every* run, including failures and dead ends. The
**registry** is curated: it holds named models, each with versions, and each
version sits in a **stage**. Stages encode an operational promise:

- `None` — registered but not vetted.
- `Staging` — passed tests, candidate for production.
- `Production` — currently serving traffic.
- `Archived` — retired (kept for rollback/audit).

Crucially, *promotion is a deliberate transition*, not a file copy. That
transition is where governance lives: approvals, gates, audit logs.

## 6. 🔧 From scratch: the model registry

In [5]:
@dataclass
class ModelVersion:
    name: str
    version: int
    run_id: str
    stage: str = "None"
    created_at: float = field(default_factory=time.time)
    description: str = ""


class ModelRegistry:
    """Versioned model registry with stage transitions and an audit log."""
    STAGES = {"None", "Staging", "Production", "Archived"}

    def __init__(self, root: pathlib.Path):
        self.root = pathlib.Path(root); self.root.mkdir(parents=True, exist_ok=True)
        self.index_file = self.root / "registry.json"
        self.audit_file = self.root / "audit.log"
        if not self.index_file.exists():
            self.index_file.write_text(json.dumps({"models": {}}))

    def _read(self): return json.loads(self.index_file.read_text())
    def _write(self, d): self.index_file.write_text(json.dumps(d, indent=2))
    def _audit(self, msg):
        with open(self.audit_file, "a") as f:
            f.write(f"{time.time():.0f}  {msg}\n")

    def register(self, name: str, run_id: str, description="") -> ModelVersion:
        d = self._read()
        versions = d["models"].setdefault(name, [])
        version = len(versions) + 1
        mv = ModelVersion(name, version, run_id, "None", time.time(), description)
        versions.append(asdict(mv))
        self._write(d)
        self._audit(f"REGISTER {name} v{version} from run {run_id}")
        return mv

    def transition(self, name: str, version: int, stage: str):
        assert stage in self.STAGES, f"unknown stage {stage}"
        d = self._read()
        versions = d["models"][name]
        # If promoting to Production, archive the current Production version.
        if stage == "Production":
            for v in versions:
                if v["stage"] == "Production":
                    v["stage"] = "Archived"
                    self._audit(f"AUTO-ARCHIVE {name} v{v['version']}")
        for v in versions:
            if v["version"] == version:
                old = v["stage"]; v["stage"] = stage
                self._audit(f"TRANSITION {name} v{version}: {old} -> {stage}")
        self._write(d)

    def get_production(self, name: str) -> dict | None:
        d = self._read()
        for v in d["models"].get(name, []):
            if v["stage"] == "Production":
                return v
        return None

    def list_versions(self, name: str) -> list[dict]:
        return self._read()["models"].get(name, [])


registry = ModelRegistry(WORK / "registry")
print("Registry at:", registry.root)

Registry at: /tmp/mlops_week_fsrlwtb7/registry


## 7. 🔧 The full workflow: track → register → promote → roll back

This is the canonical MLOps loop you'll automate in Week 4. Here we drive it by hand.

In [6]:
# 1. Register our champion run as model version 1, then push it to Production.
mv1 = registry.register("linreg", champion.run_id, "first sweep winner")
registry.transition("linreg", mv1.version, "Staging")
registry.transition("linreg", mv1.version, "Production")

# 2. A new, better run appears later. Register it and promote it.
rid2 = store.start_run("linreg_sweep", params={"lr": 0.3, "epochs": 300})
w, b, final2 = train_linear(0.3, 300, rid2, store); store.end_run(rid2)
mv2 = registry.register("linreg", rid2, "retrained, more epochs")
registry.transition("linreg", mv2.version, "Staging")
registry.transition("linreg", mv2.version, "Production")  # auto-archives v1

print("Current Production:", registry.get_production("linreg"))
print("\nAll versions:")
for v in registry.list_versions("linreg"):
    print(f"  v{v['version']}  stage={v['stage']:<11} run={v['run_id']}  ({v['description']})")

# 3. Suppose v2 misbehaves in production. Roll back: re-promote v1.
print("\n--- Rolling back to v1 ---")
registry.transition("linreg", 1, "Production")  # auto-archives v2
print("Production after rollback:", registry.get_production("linreg")["version"])

print("\n=== Audit log ===")
print(registry.audit_file.read_text())

Current Production: {'name': 'linreg', 'version': 2, 'run_id': '43ff9aa45792', 'stage': 'Production', 'created_at': 1779852500.960563, 'description': 'retrained, more epochs'}

All versions:
  v1  stage=Archived    run=20b1a6839cbd  (first sweep winner)
  v2  stage=Production  run=43ff9aa45792  (retrained, more epochs)

--- Rolling back to v1 ---
Production after rollback: 1

=== Audit log ===
1779852501  REGISTER linreg v1 from run 20b1a6839cbd
1779852501  TRANSITION linreg v1: None -> Staging
1779852501  TRANSITION linreg v1: Staging -> Production
1779852501  REGISTER linreg v2 from run 43ff9aa45792
1779852501  TRANSITION linreg v2: None -> Staging
1779852501  AUTO-ARCHIVE linreg v1
1779852501  TRANSITION linreg v2: Staging -> Production
1779852501  AUTO-ARCHIVE linreg v2
1779852501  TRANSITION linreg v1: Archived -> Production



Look at the audit log: every transition is recorded with a timestamp. That log is
the difference between *"someone changed prod, no idea when or why"* and a
governable system. Rollback is just another transition — no scrambling for old
weights, because every version is still addressable.

## 8. 🏭 In practice: MLflow

Everything you built has a near-exact MLflow counterpart:

| You built | MLflow |
|-----------|--------|
| `store.start_run / log_param / log_metric / log_artifact` | `mlflow.start_run()`, `mlflow.log_param/metric/artifact` |
| `store.search_runs` + manual table | `mlflow.search_runs()` → pandas DataFrame, plus a web UI |
| `registry.register` | `mlflow.register_model()` |
| `registry.transition` | `MlflowClient().transition_model_version_stage()` |
| `audit.log` | MLflow's backend DB + UI history |

The real MLflow adds: a database backend (Postgres) for scale, a web UI, a REST
API so remote machines log to a central server, artifact storage on S3/GCS, and a
`pyfunc` model-flavor abstraction. But the *mental model* is exactly what you just
implemented. When MLflow does something surprising, come back to this notebook —
the surprise is almost always a feature layered on top of these primitives.

```python
# The equivalent in real MLflow (do not run here — shown for mapping):
# import mlflow
# with mlflow.start_run() as run:
#     mlflow.log_param("lr", 0.3)
#     mlflow.log_metric("mse", loss, step=epoch)
#     mlflow.log_artifact("weights.json")
# mlflow.register_model(f"runs:/{run.info.run_id}/model", "linreg")
```

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Add `log_metric` *history plotting*: write a function that, given a run and metric name, prints an ASCII sparkline of the metric over steps. (The MLflow UI's metric charts are just this with pixels.)
2. Implement `search_runs(filter_fn)` that takes a predicate, e.g. `lambda r: r.params['lr'] > 0.1 and r.metrics['final_mse'][-1][1] < 0.02`. This is MLflow's query string, but as real Python.
3. Add an *approval gate* to `transition`: promotion to Production requires a `approved_by` argument, and the transition is refused (and audited as DENIED) without it.
4. Two runs produce bitwise-identical weights artifacts. Use the content hashing from Week 0 to make the registry *dedupe* them, storing the bytes once. What does this save at scale?
5. Currently a run can be registered to Production without ever passing Staging. Add a state machine that forbids None → Production directly. Where in a real org would this rule live?

---

## ✅ Recap — Week 1

- Tracking durably records inputs and outputs of every run so you can find and compare them later.
- A registry is the curated, operational layer: named models, versions, and stages.
- Stage transitions (with an audit log) are where governance and rollback live.
- We built a working file-backed tracker + registry; MLflow is the same model with a DB, UI, and scale.
- Picking a champion run programmatically is the seed of automated promotion (Week 4).

### Coming up next

**Week 2 — Model Packaging & Containerization.** We have a chosen model. Now we make it *portable*: we'll dissect what a container actually is, build a from-scratch packaging format that bundles model + env + code, and then map it onto Docker.